In [145]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import pyarrow

In [93]:

# 1. Load the datasets
criminality = pd.read_csv('C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/criminality.csv')
disruptions = pd.read_csv('C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/daily_disruptions_weather.csv')
proximity = pd.read_csv('C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/proximity_compiled_wide.csv')
socioeconomic = pd.read_csv('C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/socioeconomic_all.csv')


C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28984\3970421285.py:3: DtypeWarning: Columns (10,11,12,13,14,15,16,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  disruptions = pd.read_csv('C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/daily_disruptions_weather.csv')
C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28984\3970421285.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  proximity = pd.read_csv('C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/proximity_compiled_wide.csv')


In [94]:
# 2. Standardize Geographic Columns
# Cleaning column names for any leading/trailing spaces
for df in [criminality, disruptions, proximity, socioeconomic]:
    df.columns = df.columns.str.strip()

criminality.rename(columns={'RegionCode': 'region_code', 'MunicipalityName': 'municipality'}, inplace=True)
proximity.rename(columns={'Region_Code': 'region_code', 'Municipality_Name': 'municipality'}, inplace=True)



In [95]:
# Group by municipality and calculate the mean for all numeric columns
proximity_avg = proximity.groupby('municipality').mean(numeric_only=True)
criminality_avg = criminality.groupby('municipality').mean(numeric_only=True)

# Remove municipality from the index
proximity_avg = proximity_avg.reset_index()
criminality_avg = criminality_avg.reset_index()

In [96]:
# Path to your file
file_path = "C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/geospatial/wijkenbuurten_2024.gpkg"

# Read the file
gdf = gpd.read_file(file_path)

buurtcode = gdf[["buurtcode", "gemeentenaam", "geometry"]].rename(columns={"buurtcode" : "neighbourhood_code", "gemeentenaam" : "municipality"})

c:\Users\EduardCP\Documents\GitHub\MasterThesis\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'wijkenbuurten_2024.gpkg': 'buurten' (default), 'gemeenten', 'wijken'. Specify layer parameter to avoid this warning.
  result = read_func(


In [ ]:
socioeconomic = socioeconomic.merge(buurtcode, on="neighbourhood_code", how="left")
socioeconomic = gpd.GeoDataFrame(socioeconomic, geometry='geometry')

cols_to_keep = socioeconomic.select_dtypes(include='number').columns.tolist()
cols_to_keep.extend(['municipality', 'geometry'])

# 2. Filter the dataframe to only these columns
socioeconomic_filtered = socioeconomic[cols_to_keep]

# 3. Use dissolve to group by Municipality AND Year
# This calculates the mean for numbers AND merges the polygons
socioeconomic_avg = socioeconomic_filtered.dissolve(
    by=['municipality', 'year'], 
    aggfunc='mean'
)

In [133]:
socioeconomic_avg = socioeconomic_avg.reset_index(drop=False)

full_geo = socioeconomic_avg.merge(
    criminality_avg, 
    on="municipality", 
    how="left"
).merge(
    proximity_avg.drop(columns=["year"]),
    on="municipality", 
    how="left"
)


In [134]:
df_coordinates = pd.read_csv('C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/geospatial/stations-2023-09-nl.csv')
unique_stations = df_coordinates[['name_long', 'geo_lat', 'geo_lng']].drop_duplicates().rename(columns={"station" : "name_long"})

from shapely.geometry import Point

# Convert CSV to GeoDataFrame
stations_gdf = gpd.GeoDataFrame(
    unique_stations, 
    geometry=gpd.points_from_xy(unique_stations.geo_lng, unique_stations.geo_lat),
    crs="EPSG:4326" # Standard GPS Lat/Lon
)

# CRITICAL: Match the CRS of your municipality data (likely EPSG:28992)
stations_gdf = stations_gdf.to_crs(full_geo.crs)

In [135]:
# 'within' finds points inside the polygons
stations_mapped = gpd.sjoin(
    stations_gdf[['name_long', 'geometry']], 
    full_geo[['municipality', 'geometry']].drop_duplicates(subset='municipality'), 
    predicate='within', 
    how='inner'
)

# Drop the geometry from this lookup to make the final merge cleaner
station_lookup = stations_mapped.drop(columns='geometry')

In [136]:
# This creates the repetition you asked for
final_dataset = full_geo.merge(
    station_lookup[['municipality', 'name_long']], 
    on='municipality', 
    how='left'
)

# Rename 'name_long' to 'station' if that's your preferred variable name
final_dataset = final_dataset.rename(columns={'name_long': 'station'})

In [140]:
# 1. Define the final list of 28 variables based on your study goals
selected_columns = [
    # Identifiers & Spatial
    'municipality', 'station', 'year', 'geometry', 'Population',
    
    # Economic & Wealth (Using Indices where possible for better scaling)
    'SES_Score_Wealth_Avg', 'Income_Avg_Percentile_Score', 
    'Wealth_Percentile_1_40_pct', 'NetWorth_Avg_Percentile_Score',
    'PrivateHouseholds_Count',
    
    # Employment & Labor Stability
    'SES_Score_WorkHistory_Avg', 'WorkHistory_NotEmployed_Last4Y_pct',
    'WorkHistory_Retired_pct', 'WorkHistory_ConstantlyEmployed_Last4Y_pct',
    
    # Education
    'SES_Score_Education_Avg', 'Edu_Low_pct', 'Edu_High_Total_pct',
    
    # Crime & Social Safety (Prioritizing Rates and Vandalism)
    'TotalPropertyDamageAndViolence_Index', 'TotalVandalism',
    'TotalViolentAndSexualCrimes', 'BicycleTheft', 'TotalPropertyCrimes_Rate',
    
    # Infrastructure & Proximity (Reliance on the network)
    'Dist_Train_Station_Total', 'Dist_Major_Transfer_Station',
    'Dist_Highway_Entrance', 'Dist_Supermarket', 
    'Dist_GP_Surgery', 'Dist_Secondary_Total'
]

# 2. Filter the dataframe
# Note: Using 'errors=ignore' just in case a specific column was renamed earlier
df = final_dataset[final_dataset.columns.intersection(selected_columns)]

In [142]:
# 1. Prepare the lookup table from your previously created df
# We drop 'municipality', 'year', and 'geometry' to keep the merge clean, 
# keeping only 'station' and the numeric indicators.
lookup_df = df.drop(columns=['municipality', 'geometry'], errors='ignore')

# Create a year var for the distruption dataset
disruptions['Date'] = pd.to_datetime(disruptions['Date'])
disruptions['year'] = disruptions['Date'].dt.year

# 2. Calculate global averages for imputation (for international/missing stations)
# We exclude 'year' and 'station' from the mean calculation
global_averages = lookup_df.select_dtypes(include='number').drop(columns=['year'], errors='ignore').mean()

# 3. Join for the SOURCE station
disruptions_full = disruptions.merge(
    lookup_df.add_prefix('source_'), 
    left_on=['source', 'year'], 
    right_on=['source_station', 'source_year'], 
    how='left'
).drop(columns=['source_station', 'source_year'])

# 4. Join for the TARGET station
disruptions_full = disruptions_full.merge(
    lookup_df.add_prefix('target_'), 
    left_on=['target', 'year'], 
    right_on=['target_station', 'target_year'], 
    how='left'
).drop(columns=['target_station', 'target_year'])



In [149]:
# 5. Impute missing values (International Stations)
# We fill NaNs using the global averages calculated in step 2
source_cols = [c for c in disruptions_full.columns if c.startswith('source_')]
target_cols = [c for c in disruptions_full.columns if c.startswith('target_')]

# Apply the averages to the respective source and target columns
disruptions_full[source_cols] = disruptions_full[source_cols].fillna(global_averages.add_prefix('source_'))
disruptions_full[target_cols] = disruptions_full[target_cols].fillna(global_averages.add_prefix('target_'))


# 1. Replace empty strings/spaces with NaN across the whole dataframe
# This fixes the '      ' error in the 'DR' column and others
disruptions_full = disruptions_full.replace(r'^\s*$', pd.NA, regex=True)

# 2. Specifically fix the "DR" column if it's supposed to be numeric
# We use errors='coerce' to turn anything unparseable into NaN
if 'DR' in disruptions_full.columns:
    disruptions_full['DR'] = pd.to_numeric(disruptions_full['DR'], errors='coerce')

# 3. Handle the geometry column (if it still exists)
# Parquet doesn't like 'geometry' objects unless you use a specific GeoParquet writer.
# If you don't need the maps in this specific file, drop it:
if 'geometry' in disruptions_full.columns:
    disruptions_full = disruptions_full.drop(columns=['geometry'])


# 6. Save to Parquet
output_path = 'C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/final_dataset.parquet'
disruptions_full.to_parquet(output_path, engine='fastparquet', index=False)

print(f"Final dataset saved successfully! Shape: {disruptions_full.shape}")

Final dataset saved successfully! Shape: (654436, 70)


In [ ]:
# 1. Take a random sample of 10,000 rows
# 'random_state' ensures that if you run this again, you get the exact same 10,000 rows
subset_df = disruptions_full.sample(n=10000, random_state=42)

if 'geometry' in subset_df.columns:
    subset_df = subset_df.drop(columns=['geometry'])

# 3. Save to CSV
subset_path = 'C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/subset_10k_disruptions.csv'

subset_df.to_csv(subset_path, index=False, sep=';', encoding='utf-8-sig')

print(f"Subset of {len(subset_df)} rows saved to: {subset_path}")

Subset of 10000 rows saved to: C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/subset_10k_disruptions.csv
